## Let's mount the drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cloning the github to access the dataset

In [ ]:
!git clone https://github.com/HumBug-Mosquito/HumBugDB.git
%cd HumBugDB

Cloning into 'HumBugDB'...
remote: Enumerating objects: 1046, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 1046 (delta 50), reused 18 (delta 18), pack-reused 991 (from 1)
Receiving objects: 100% (1046/1046), 21.59 MiB | 35.55 MiB/s, done.
Resolving deltas: 100% (502/502), done.
/content/HumBugDB


## Create folders in the mounted Drive

In [ ]:
!wget https://zenodo.org/record/4904800/files/humbugdb_neurips_2021_1.zip
!wget https://zenodo.org/record/4904800/files/humbugdb_neurips_2021_2.zip
!wget https://zenodo.org/record/4904800/files/humbugdb_neurips_2021_3.zip
!wget https://zenodo.org/record/4904800/files/humbugdb_neurips_2021_4.zip

--2026-08-05 06:57:22--  https://zenodo.org/record/4904800/files/humbugdb_neurips_2021_1.zip
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 137.138.52.235, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/4904800/files/humbugdb_neurips_2021_1.zip [following]
--2026-08-05 06:57:22--  https://zenodo.org/records/4904800/files/humbugdb_neurips_2021_1.zip
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 1042703724 (994M) [application/octet-stream]
Saving to: ‘humbugdb_neurips_2021_1.zip’

humbugdb_neurips_20 100%[===================>] 994.40M  2.17MB/s    in 6m 41s  

2026-08-05 07:04:04 (2.48 MB/s) - ‘humbugdb_neurips_2021_1.zip’ saved [1042703724/1042703724]

--2026-08-05 07:04:04--  https://zenodo.org/record/4904800/files/humbugdb_neurips_2021_2.zip
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 188

## Back up the raw zip files to the Drive

In [ ]:

!cp humbugdb_neurips_2021_*.zip "/content/drive/MyDrive/HumBugDB_Project/raw_audio/"

cp: cannot stat 'humbugdb_neurips_2021_*.zip': No such file or directory


## Create the target directory in the cloned repo

In [ ]:
!mkdir -p data/audio

# 3. Unzip all files directly into the audio folder

In [ ]:
!unzip -q -j "/content/drive/MyDrive/HumBugDB_Project/raw_audio/humbugdb_neurips_2021_*.zip" -d data/audio/

In [ ]:
!uv pip install -q torchaudio
!uv pip install -q torchcodec
!uv pip install -q torch
!uv pip install -q datasets[audio]
!uv pip install -q protobuf

In [ ]:
import pandas as pd
metadata = pd.read_csv('data/metadata/neurips_2021_zenodo_0_0_1.csv')


In [ ]:
metadata

,id,length,name,sample_rate,record_datetime,sound_type,species,gender,fed,plurality,age,method,mic_type,device_type,country,district,province,place,location_type
0,199980,1121.880000,background.wav,8000,08-09-16 08:00,background,NaN,NaN,NaN,NaN,NaN,NaN,phone,Alcatel 4009X,USA,Georgia,Atlanta,"CDC insect cultures, Atlanta",culture
1,53,0.463456,CDC_Ae-aegypti_labelled_800.wav,8000,08-09-16 08:00,mosquito,ae aegypti,NaN,NaN,Single,NaN,NaN,phone,Alcatel 4009X,USA,Georgia,Atlanta,"CDC insect cultures, Atlanta",culture
2,57,0.170249,CDC_Ae-aegypti_labelled_800.wav,8000,08-09-16 08:00,mosquito,ae aegypti,NaN,NaN,Single,NaN,NaN,phone,Alcatel 4009X,USA,Georgia,Atlanta,"CDC insect cultures, Atlanta",culture
3,61,0.104041,CDC_Ae-aegypti_labelled_800.wav,8000,08-09-16 08:00,mosquito,ae aegypti,NaN,NaN,Single,NaN,NaN,phone,Alcatel 4009X,USA,Georgia,Atlanta,"CDC insect cultures, Atlanta",culture
4,69,0.274290,CDC_Ae-aegypti_labelled_800.wav,8000,08-09-16 08:00,mosquito,ae aegypti,NaN,NaN,Single,NaN,NaN,phone,Alcatel 4009X,USA,Georgia,Atlanta,"CDC insect cultures, Atlanta",culture
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9290,4339,1.812731,larvae_#8-11_rec3.wav,44100,01-07-18 12:00,mosquito,NaN,NaN,NaN,NaN,NaN,LC,telinga,olympus,Thailand,Sai Yok District,Kanchanaburi Province,field site near Pu Teuy Village,cup
9291,202190,2.122423,larvae_#8-11_rec3.wav,44100,01-07-18 12:00,background,NaN,NaN,NaN,NaN,NaN,LC,telinga,olympus,Thailand,Sai Yok District,Kanchanaburi Province,field site near Pu Teuy Village,cup
9292,202191,1.891186,larvae_#8-11_rec3.wav,44100,01-07-18 12:00,background,NaN,NaN,NaN,NaN,NaN,LC,telinga,olympus,Thailand,Sai Yok District,Kanchanaburi Province,field site near Pu Teuy Village,cup
9293,4340,7.527583,larvae_#8-11_rec3.wav,44100,01-07-18 12:00,mosquito,NaN,NaN,NaN,NaN,NaN,LC,telinga,olympus,Thailand,Sai Yok District,Kanchanaburi Province,field site near Pu Teuy Village,cup


In [ ]:
import pandas as pd
import librosa
import numpy as np
import os

# Load the full metadata
metadata = pd.read_csv('data/metadata/neurips_2021_zenodo_0_0_1.csv')

spectrograms = []
labels = []

# Convert ALL files in the metadata, not just a subset
print(f"Processing all {len(metadata)} audio files in the dataset...")

for index, row in metadata.iterrows():
    file_id = row['id']
    sound_type = row['sound_type']

    file_path = f'data/audio/{file_id}.wav'

    if os.path.exists(file_path):
        # Load and convert to log-mel spectrogram
        y, sr = librosa.load(file_path, sr=8000)
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
        log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)

        spectrograms.append(log_mel_spec)

        # Label 1 for mosquito, 0 for background noise
        if sound_type == 'mosquito':
            labels.append(1)
        else:
            labels.append(0)

# Convert to NumPy arrays and save to Google Drive
X_data = np.array(spectrograms)
y_data = np.array(labels)

save_path = '/content/drive/MyDrive/HumBugDB_Project/processed_features'
np.save(f'{save_path}/X_spectrograms_full.npy', X_data)
np.save(f'{save_path}/y_labels_full.npy', y_data)

In [ ]:
from datasets import Dataset, Audio


metadata['audio'] = metadata['id'].apply(lambda x: f'data/audio/{x}.wav')

# Create a Hugging Face Dataset from the pandas DataFrame
ds = Dataset.from_pandas(metadata)

ds = ds.cast_column("audio", Audio(sampling_rate=8000))

print("Dataset created successfully!")
display(ds)

Dataset created successfully!


Dataset({
    features: ['id', 'length', 'name', 'sample_rate', 'record_datetime', 'sound_type', 'species', 'gender', 'fed', 'plurality', 'age', 'method', 'mic_type', 'device_type', 'country', 'district', 'province', 'place', 'location_type', 'audio'],
    num_rows: 9295
})

In [ ]:
# Example: Access the first element to verify loading
example = ds[0]
print(f"Example Audio Shape: {example['audio']['array'].shape}")
print(f"Label: {example['sound_type']}")

Example Audio Shape: (8975040,)
Label: background


In [ ]:
!hf auth login

Hint: A new version of huggingface_hub (1.26.0) is available! You are using version 1.23.0.
To update, run: hf update
? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: W978-VD4K

    Waiting for authorization...W978-VD4K
......
Token is valid.
The token `oauth-BigTrev89` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-BigTrev89`
Note: This token will be refreshed automatically when it expires.


### 2. Push to Hub
This will upload the metadata and the audio files. Once uploaded, the Hugging Face dataset viewer will automatically provide a play button for the audio samples.

In [ ]:
# Replace 'your-username/humbugdb-audio' with your desired repository name
ds.push_to_hub("BigTrev89/humbugdb-audio")

Uploading the dataset shards:   0%|          | 0/9 [00:00<?, ? shards/s]

Map:   0%|          | 0/1033 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   0%|          |  529kB /  819MB            

CommitInfo(commit_url='https://huggingface.co/datasets/BigTrev89/humbugdb-audio/commit/fbb8ee931e2816e5e5d534cb7c6f1d267163464d', commit_message='Upload dataset', commit_description='', oid='fbb8ee931e2816e5e5d534cb7c6f1d267163464d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/BigTrev89/humbugdb-audio', endpoint='https://huggingface.co', repo_type='dataset', repo_id='BigTrev89/humbugdb-audio'), pr_revision=None, pr_num=None)